# Visual Question Answering on the Synthetic SHAPES Dataset

**A story-driven exploration**: from a naive end-to-end baseline that fails,
to a hybrid CNN-perception + symbolic-reasoning pipeline that succeeds.

---

The SHAPES dataset consists of 30×30 RGB images containing simple geometric
shapes (circles, squares, triangles) in three colours (red, green, blue),
arranged on a 3×3 grid. Each image is paired with a compositional query
expressed in a parenthesised prefix notation (e.g. `(is blue (left_of red))`)
and a boolean answer. Our goal: predict the correct answer for unseen images
and queries.

---
## Section 1 — Setup

Before we dive into modelling, we load all our tools, define the vocabulary
of the dataset, and take a first look at the images across all four training
splits (tiny → small → medium → large).

In [ ]:
# ── Imports ─────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm
import os, warnings

warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# ── Constants ──────────────────────────────────────────
SHAPES = ['circle', 'square', 'triangle']
COLORS = ['red', 'green', 'blue']
SIZES  = ['tiny', 'small', 'med', 'large']

SHAPE_TO_IDX = {s: i for i, s in enumerate(SHAPES)}
COLOR_TO_IDX = {c: i for i, c in enumerate(COLORS)}

print(f"Shapes: {SHAPES}")
print(f"Colors: {COLORS}")
print(f"Sizes:  {SIZES}")

In [ ]:
# ── Data-Loading Helpers ───────────────────────────────
def load_split(size):
    """Load a training split by size name (tiny/small/med/large)."""
    images = np.load(f'train.{size}.input.npy')
    with open(f'train.{size}.query') as f:
        queries = [l.strip() for l in f]
    # The large split has a differently-named output file
    out_path = f'train_{size}.output' if os.path.exists(f'train_{size}.output') else f'train.{size}.output'
    with open(out_path) as f:
        labels = [l.strip() == 'true' for l in f]
    return images, queries, labels


def load_test():
    """Load the shared test split."""
    images = np.load('test.input.npy')
    with open('test.query') as f:
        queries = [l.strip() for l in f]
    with open('test.output') as f:
        labels = [l.strip() == 'true' for l in f]
    return images, queries, labels


# Quick sanity check
for sz in SIZES:
    imgs, qs, ls = load_split(sz)
    print(f"  {sz:>6}: {len(imgs):>6} images, {len(qs):>6} queries, {len(ls):>6} labels")
test_images, test_queries, test_labels = load_test()
print(f"  {'test':>6}: {len(test_images):>6} images, {len(test_queries):>6} queries, {len(test_labels):>6} labels")

In [ ]:
# ── Visualise Sample Images ───────────────────────────
fig, axes = plt.subplots(len(SIZES), 8, figsize=(16, 8))
for row, sz in enumerate(SIZES):
    imgs, _, _ = load_split(sz)
    idxs = np.random.choice(len(imgs), 8, replace=False)
    for col, idx in enumerate(idxs):
        axes[row, col].imshow(imgs[idx])
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(sz, fontsize=14, rotation=0, labelpad=50)
fig.suptitle('Sample 30×30 Images from Each Training Split', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## Section 2 — Baseline Approach (End-to-End Neural)

Our first instinct was to treat this as a standard deep-learning problem —
encode the image, encode the text, fuse them, and learn the answer directly.

This is the most natural starting point: if we throw enough data at a neural
network, surely it can learn the mapping from (image, query) → answer?

We build a simple but honest baseline:
- **Image encoder**: a small CNN that produces a 128-d vector
- **Text encoder**: bag-of-words over the query vocabulary → 64-d vector
- **Fusion MLP**: concatenate both vectors → two linear layers → binary prediction

In [ ]:
# ── Baseline: Build Vocabulary ─────────────────────────
train_imgs_large, train_queries_large, train_labels_large = load_split('large')

def build_vocab(queries):
    """Build a token → index vocabulary from all queries."""
    tokens = set()
    for q in queries:
        for ch in q:
            if ch in '()':
                tokens.add(ch)
        for tok in q.replace('(', ' ').replace(')', ' ').split():
            tokens.add(tok)
    vocab = {t: i for i, t in enumerate(sorted(tokens))}
    return vocab

vocab = build_vocab(train_queries_large + test_queries)
print(f"Vocabulary size: {len(vocab)}")
print(f"Tokens: {sorted(vocab.keys())}")

In [ ]:
# ── Baseline: Model Definition ─────────────────────────
class BaselineImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)  # (B, 128)


class BaselineTextEncoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.fc = nn.Linear(vocab_size, 64)

    def forward(self, bow):
        return torch.relu(self.fc(bow))  # (B, 64)


class BaselineVQA(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.img_enc = BaselineImageEncoder()
        self.txt_enc = BaselineTextEncoder(vocab_size)
        self.classifier = nn.Sequential(
            nn.Linear(128 + 64, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, img, bow):
        iv = self.img_enc(img)
        tv = self.txt_enc(bow)
        return self.classifier(torch.cat([iv, tv], dim=1)).squeeze(-1)


print("Baseline model architecture:")
print(BaselineVQA(len(vocab)))

In [ ]:
# ── Baseline: Dataset & Training ──────────────────────
class VQADataset(Dataset):
    def __init__(self, images, queries, labels, vocab):
        self.images = torch.from_numpy(images).permute(0, 3, 1, 2).float() / 255.0
        self.labels = torch.tensor(labels, dtype=torch.float32)
        # Build bag-of-words
        n = len(queries)
        vs = len(vocab)
        self.bow = torch.zeros(n, vs)
        for i, q in enumerate(queries):
            toks = q.replace('(', ' ( ').replace(')', ' ) ').split()
            for t in toks:
                if t in vocab:
                    self.bow[i, vocab[t]] += 1.0

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.bow[idx], self.labels[idx]


train_ds = VQADataset(train_imgs_large, train_queries_large, train_labels_large, vocab)
test_ds  = VQADataset(test_images, test_queries, test_labels, vocab)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

baseline_model = BaselineVQA(len(vocab)).to(DEVICE)
optimizer = optim.Adam(baseline_model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

baseline_losses = []
baseline_test_accs = []

for epoch in range(1, 21):
    baseline_model.train()
    total_loss = 0
    n_samples = 0
    for imgs, bows, labs in train_loader:
        imgs, bows, labs = imgs.to(DEVICE), bows.to(DEVICE), labs.to(DEVICE)
        logits = baseline_model(imgs, bows)
        loss = criterion(logits, labs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)
    avg_loss = total_loss / n_samples
    baseline_losses.append(avg_loss)

    # Test accuracy
    baseline_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, bows, labs in test_loader:
            imgs, bows, labs = imgs.to(DEVICE), bows.to(DEVICE), labs.to(DEVICE)
            preds = (torch.sigmoid(baseline_model(imgs, bows)) > 0.5).float()
            correct += (preds == labs).sum().item()
            total += labs.size(0)
    test_acc = correct / total
    baseline_test_accs.append(test_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>2}/20  Loss {avg_loss:.4f}  Test Acc {test_acc:.4f}")

In [ ]:
# ── Baseline: Results Plots ────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, 21), baseline_losses, 'b-o', markersize=3)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.set_title('Baseline Training Loss'); ax1.grid(True, alpha=0.3)

ax2.plot(range(1, 21), baseline_test_accs, 'r-o', markersize=3)
ax2.axhline(0.5, color='gray', ls='--', label='Random guess')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title(f'Baseline Test Accuracy (final: {baseline_test_accs[-1]:.3f})')
ax2.legend(); ax2.grid(True, alpha=0.3); ax2.set_ylim(0.3, 0.8)

plt.tight_layout()
plt.show()
print(f"\nFinal baseline test accuracy: {baseline_test_accs[-1]:.4f} ({int(baseline_test_accs[-1]*1024)}/1024)")

### Why the Baseline Fails

The end-to-end neural model hovers around **55–65% accuracy** — barely above
the ~50% random baseline. The model has no way to decompose the question:
it cannot handle compositional queries like `(is blue (left_of (right_of square)))`
end-to-end without understanding spatial structure.

The query language is **recursive** — operators nest arbitrarily — and the
correct answer depends on the *spatial arrangement* of objects on the grid.
A bag-of-words text encoder destroys word order entirely, and even a more
sophisticated text encoder would struggle because the reasoning is
**compositional**: each operator's output feeds the next.

**Key insight**: the *perception* task (which shapes/colours are where) is
simple, but the *reasoning* task (evaluating nested spatial relations) is
symbolic in nature. Mixing them into one model forces the network to learn
both — and it learns neither well.

This motivates a **hybrid approach**: let a CNN handle low-level perception
per grid cell, and let hand-crafted symbolic rules handle all the reasoning.

---
## Section 3 — Our Hybrid Pipeline (The Real Approach)

Instead of learning everything end-to-end, we decompose the problem into
two clean stages:

1. **Visual Perception (CNN)**: Split each 30×30 image into a 3×3 grid of
   10×10 cells. Classify each non-empty cell into shape ∈ {circle, square,
   triangle} and colour ∈ {red, green, blue}.

2. **Symbolic Reasoning**: Parse the query into an AST, then recursively
   evaluate it over the detected objects using spatial relation rules.

This design means the CNN only has to solve a *tiny* 10×10 classification
problem, and the reasoning is handled by deterministic code that is
guaranteed correct.

In [ ]:
# ── Cell Extraction ────────────────────────────────────
def extract_cells(image):
    """Split a 30x30 image into 9 cells of 10x10 (row-major order)."""
    cells = []
    for row in range(3):
        for col in range(3):
            cell = image[row * 10:(row + 1) * 10, col * 10:(col + 1) * 10, :]
            cells.append(cell)
    return cells


# Visualise one image split into its 3x3 grid
sample_img = train_imgs_large[0]
sample_cells = extract_cells(sample_img)

fig, axes = plt.subplots(1, 10, figsize=(18, 2))
axes[0].imshow(sample_img)
axes[0].set_title('Full image', fontsize=9)
axes[0].axis('off')

for i in range(9):
    axes[i + 1].imshow(sample_cells[i])
    r, c = divmod(i, 3)
    axes[i + 1].set_title(f'({r},{c})', fontsize=9)
    axes[i + 1].axis('off')

fig.suptitle('Image → 3×3 grid of 10×10 cells', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Heuristic Cell Labeller ────────────────────────────
def heuristic_label_cell(cell):
    """
    Classify a 10x10x3 cell by colour and shape using pixel analysis.
    Returns (shape, colour) or (None, None) for empty cells.
    
    Logic:
      - mask = pixels where sum(RGB) > 10
      - colour: argmax of mean RGB over masked pixels
      - shape: row-width profile analysis
          triangle: monotonically non-decreasing widths with range > 1
          square:   fill_ratio >= 0.90
          circle:   everything else
    """
    mask = cell.sum(axis=2) > 10
    n_pixels = int(mask.sum())
    if n_pixels < 5:
        return None, None

    # Colour: dominant RGB channel
    mean_rgb = cell[mask].astype(np.float32).mean(axis=0)
    color = COLORS[int(np.argmax(mean_rgb))]

    # Shape: row-width profile analysis
    row_widths = [int(mask[r].sum()) for r in range(10) if mask[r].sum() > 0]
    if len(row_widths) < 2:
        return None, None

    widths = np.array(row_widths)

    # Triangle: monotonically non-decreasing with non-trivial range
    if np.all(np.diff(widths) >= 0) and widths.max() - widths.min() > 1:
        return 'triangle', color

    # Fill-ratio boundary: >= 0.90 => square, else circle
    fill_ratio = n_pixels / (len(widths) * int(widths.max()))
    if fill_ratio >= 0.90:
        return 'square', color

    return 'circle', color


def auto_label_cells(images):
    """
    Label all non-empty cells in *images* using the heuristic.
    Returns (cells_array, shape_labels, color_labels).
    """
    all_cells, all_shapes, all_colors = [], [], []
    for img in images:
        for cell in extract_cells(img):
            shape, color = heuristic_label_cell(cell)
            if shape is not None:
                all_cells.append(cell)
                all_shapes.append(SHAPE_TO_IDX[shape])
                all_colors.append(COLOR_TO_IDX[color])
    return np.array(all_cells), np.array(all_shapes), np.array(all_colors)


print("Heuristic labeller defined.")

In [ ]:
# ── Visualise Heuristic Labels ─────────────────────────
vis_cells, vis_shapes, vis_colors = auto_label_cells(train_imgs_large[:20])
n_show = min(40, len(vis_cells))

fig, axes = plt.subplots(4, 10, figsize=(18, 8))
color_map = {'red': '#e74c3c', 'green': '#27ae60', 'blue': '#2980b9'}

for i in range(n_show):
    ax = axes[i // 10, i % 10]
    ax.imshow(vis_cells[i])
    s_name = SHAPES[vis_shapes[i]]
    c_name = COLORS[vis_colors[i]]
    ax.set_title(f'{c_name} {s_name}', fontsize=8, color=color_map[c_name])
    for spine in ax.spines.values():
        spine.set_edgecolor(color_map[c_name])
        spine.set_linewidth(3)
    ax.set_xticks([]); ax.set_yticks([])

# Hide unused axes
for i in range(n_show, 40):
    axes[i // 10, i % 10].axis('off')

fig.suptitle('Heuristic Labels: First 40 Non-Empty Cells', fontsize=14)
plt.tight_layout()
plt.show()

# Distribution
print("Shape distribution:", {s: int((vis_shapes == i).sum()) for i, s in enumerate(SHAPES)})
print("Color distribution:", {c: int((vis_colors == i).sum()) for i, c in enumerate(COLORS)})

In [ ]:
# ── CellClassifier CNN ─────────────────────────────────
class CellClassifier(nn.Module):
    """Shallow CNN with two heads for shape + colour."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.shape_head = nn.Linear(64, len(SHAPES))
        self.color_head = nn.Linear(64, len(COLORS))

    def forward(self, x):
        feat = self.features(x).view(x.size(0), -1)
        return self.shape_head(feat), self.color_head(feat)


class CellDataset(Dataset):
    def __init__(self, cells, shapes, colors):
        self.cells = torch.from_numpy(cells).permute(0, 3, 1, 2).float() / 255.0
        self.shapes = torch.from_numpy(shapes).long()
        self.colors = torch.from_numpy(colors).long()

    def __len__(self):
        return len(self.cells)

    def __getitem__(self, idx):
        return self.cells[idx], self.shapes[idx], self.colors[idx]


print("CellClassifier architecture:")
print(CellClassifier())

In [ ]:
# ── Train CNN ──────────────────────────────────────────
def train_cnn(cells, shapes, colors, epochs=30, batch_size=256, lr=1e-3, verbose=True):
    """Train the cell classifier. Returns (model, history_dict)."""
    dataset = CellDataset(cells, shapes, colors)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model = CellClassifier().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {'loss': [], 'shape_acc': [], 'color_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = correct_s = correct_c = total = 0
        for bx, bs, bc in loader:
            bx, bs, bc = bx.to(DEVICE), bs.to(DEVICE), bc.to(DEVICE)
            sl, cl = model(bx)
            loss = criterion(sl, bs) + criterion(cl, bc)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * bx.size(0)
            correct_s += (sl.argmax(1) == bs).sum().item()
            correct_c += (cl.argmax(1) == bc).sum().item()
            total += bx.size(0)

        history['loss'].append(total_loss / total)
        history['shape_acc'].append(correct_s / total)
        history['color_acc'].append(correct_c / total)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"  Epoch {epoch:>2}/{epochs}  Loss {history['loss'][-1]:.4f}  "
                  f"Shape {history['shape_acc'][-1]:.4f}  Color {history['color_acc'][-1]:.4f}")

    model.eval()
    return model, history


def plot_training_history(history, title='CNN Training'):
    """Side-by-side loss + accuracy plots."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['loss']) + 1)

    ax1.plot(epochs, history['loss'], 'b-')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} — Loss'); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history['shape_acc'], 'g-', label='Shape Acc')
    ax2.plot(epochs, history['color_acc'], 'r-', label='Color Acc')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{title} — Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.show()


print("train_cnn() and plot_training_history() defined.")

In [ ]:
# ── Perceive Image ─────────────────────────────────────
def perceive_image(model, image):
    """Run CNN on a 30x30 image → list of {shape, color, x, y} dicts."""
    cells = extract_cells(image)
    non_empty, positions = [], []
    for idx, cell in enumerate(cells):
        if cell.sum(axis=2).max() > 10:
            non_empty.append(cell)
            row, col = divmod(idx, 3)
            positions.append((col, row))  # x=col, y=row
    if not non_empty:
        return []
    batch = (torch.from_numpy(np.array(non_empty))
             .permute(0, 3, 1, 2).float() / 255.0).to(DEVICE)
    with torch.no_grad():
        sl, cl = model(batch)
        sp = sl.argmax(1).cpu().numpy()
        cp = cl.argmax(1).cpu().numpy()
    return [{'shape': SHAPES[sp[i]], 'color': COLORS[cp[i]],
             'x': positions[i][0], 'y': positions[i][1]}
            for i in range(len(non_empty))]


print("perceive_image() defined.")

In [ ]:
# ── Query Parser ───────────────────────────────────────
def tokenize(query):
    """Tokenize a parenthesised prefix query."""
    tokens, i = [], 0
    while i < len(query):
        ch = query[i]
        if ch in '()':
            tokens.append(ch); i += 1
        elif ch.isspace():
            i += 1
        else:
            j = i
            while j < len(query) and query[j] not in '() \t\n':
                j += 1
            tokens.append(query[i:j]); i = j
    return tokens


def _parse(tokens, pos):
    if tokens[pos] == '(':
        pos += 1
        op = tokens[pos]; pos += 1
        args = []
        while tokens[pos] != ')':
            arg, pos = _parse(tokens, pos)
            args.append(arg)
        return (op, *args), pos + 1
    return tokens[pos], pos + 1


def parse_query(query_str):
    """Parse a parenthesised prefix query into a nested-tuple AST."""
    tree, _ = _parse(tokenize(query_str.strip()), 0)
    return tree


# Demo
demo_queries = [
    "(is blue (left_of red))",
    "(is green (right_of (above square)))",
    "(is blue (left_of (right_of square)))",
]
for q in demo_queries:
    print(f"  {q}")
    print(f"    → {parse_query(q)}\n")

In [ ]:
# ── Symbolic Executor ──────────────────────────────────
def execute(tree, objects):
    """
    Evaluate the query AST over the list of detected objects.

    Leaves return the set of matching object indices.
    Relational operators (left_of, right_of, above, below) return sets.
    'is' returns a boolean.
    """
    if isinstance(tree, str):
        return {i for i, o in enumerate(objects)
                if o['color'] == tree or o['shape'] == tree}

    op = tree[0]

    if op == 'is':
        attr = tree[1]
        result_set = execute(tree[2], objects)
        return any(objects[i]['color'] == attr or objects[i]['shape'] == attr
                   for i in result_set)

    inner = execute(tree[1], objects)
    if not inner:
        return set()

    if op == 'left_of':
        return {i for i in range(len(objects))
                if any(objects[i]['x'] < objects[j]['x'] for j in inner)}
    if op == 'right_of':
        return {i for i in range(len(objects))
                if any(objects[i]['x'] > objects[j]['x'] for j in inner)}
    if op == 'above':
        return {i for i in range(len(objects))
                if any(objects[i]['y'] < objects[j]['y'] for j in inner)}
    if op == 'below':
        return {i for i in range(len(objects))
                if any(objects[i]['y'] > objects[j]['y'] for j in inner)}

    raise ValueError(f"Unknown operator: {op}")


# Quick demo
demo_objects = [
    {'shape': 'circle', 'color': 'blue', 'x': 0, 'y': 0},
    {'shape': 'square', 'color': 'red',  'x': 2, 'y': 1},
    {'shape': 'triangle', 'color': 'green', 'x': 1, 'y': 2},
]
demo_tree = parse_query("(is blue (left_of red))")
print(f"Query:   (is blue (left_of red))")
print(f"Objects: {demo_objects}")
print(f"Result:  {execute(demo_tree, demo_objects)}")

In [ ]:
# ── Self-Training Refinement ───────────────────────────
def _build_cell_data(images):
    """Pre-compute per-image cell info with heuristic labels."""
    all_cells, all_shapes, all_colors = [], [], []
    image_cell_info = []
    for img in images:
        cells = extract_cells(img)
        img_cells = []
        for idx, cell in enumerate(cells):
            shape, color = heuristic_label_cell(cell)
            if shape is not None:
                gi = len(all_cells)
                row, col = divmod(idx, 3)
                all_cells.append(cell)
                all_shapes.append(SHAPE_TO_IDX[shape])
                all_colors.append(COLOR_TO_IDX[color])
                img_cells.append({'color': color, 'x': col, 'y': row, 'gi': gi})
        image_cell_info.append(img_cells)
    return (np.array(all_cells), np.array(all_shapes), np.array(all_colors),
            image_cell_info)


def _eval_labels(shapes_arr, image_cell_info, parsed_queries, labels):
    """Count how many images are answered correctly by current shape labels."""
    correct = 0
    for i, ici in enumerate(image_cell_info):
        objs = [{'shape': SHAPES[shapes_arr[c['gi']]],
                 'color': c['color'], 'x': c['x'], 'y': c['y']} for c in ici]
        if bool(execute(parsed_queries[i], objs)) == labels[i]:
            correct += 1
    return correct


def _single_cell_corrections(shapes_arr, image_cell_info, parsed_queries, labels):
    """For each wrong answer, if flipping exactly ONE cell fixes it, apply that flip."""
    n_fixed = 0
    for i, ici in enumerate(image_cell_info):
        objs = [{'shape': SHAPES[shapes_arr[c['gi']]],
                 'color': c['color'], 'x': c['x'], 'y': c['y']} for c in ici]
        if bool(execute(parsed_queries[i], objs)) == labels[i]:
            continue
        fixes = []
        for j, c in enumerate(ici):
            orig = objs[j]['shape']
            for ns in SHAPES:
                if ns == orig:
                    continue
                objs[j]['shape'] = ns
                if bool(execute(parsed_queries[i], objs)) == labels[i]:
                    fixes.append((c['gi'], SHAPE_TO_IDX[ns]))
                objs[j]['shape'] = orig
        if len(fixes) == 1:
            shapes_arr[fixes[0][0]] = fixes[0][1]
            n_fixed += 1
    return n_fixed


def refine_labels(cells, shapes, colors, image_cell_info,
                  parsed_queries, labels, rounds=3, cnn_epochs=25, verbose=True):
    """
    Self-training loop:
      1. Apply unique single-cell corrections from query feedback
      2. Train CNN on current labels
      3. Adopt CNN predictions where they fix remaining wrong answers
    Returns the final (model).
    """
    best_model = None

    for r in range(1, rounds + 1):
        n_fix = _single_cell_corrections(shapes, image_cell_info,
                                         parsed_queries, labels)
        acc = _eval_labels(shapes, image_cell_info, parsed_queries, labels)
        if verbose:
            print(f"    Round {r}: {n_fix} corrections → "
                  f"accuracy {acc}/{len(labels)} = {acc/len(labels):.4f}")

        model, _ = train_cnn(cells, shapes, colors,
                             epochs=cnn_epochs, verbose=False)
        best_model = model

        # CNN batch-predict
        preds = []
        for s in range(0, len(cells), 512):
            b = (torch.from_numpy(cells[s:s+512])
                 .permute(0, 3, 1, 2).float() / 255.0).to(DEVICE)
            with torch.no_grad():
                preds.append(model(b)[0].argmax(1).cpu().numpy())
        cnn_shapes = np.concatenate(preds)

        improved = shapes.copy()
        any_change = False
        for i, ici in enumerate(image_cell_info):
            objs_h = [{'shape': SHAPES[shapes[c['gi']]],
                       'color': c['color'], 'x': c['x'], 'y': c['y']}
                      for c in ici]
            if bool(execute(parsed_queries[i], objs_h)) == labels[i]:
                continue
            objs_c = [{'shape': SHAPES[cnn_shapes[c['gi']]],
                       'color': c['color'], 'x': c['x'], 'y': c['y']}
                      for c in ici]
            if bool(execute(parsed_queries[i], objs_c)) == labels[i]:
                for c in ici:
                    improved[c['gi']] = cnn_shapes[c['gi']]
                any_change = True

        imp_acc = _eval_labels(improved, image_cell_info, parsed_queries, labels)
        if imp_acc > acc:
            shapes[:] = improved
            if verbose:
                print(f"             CNN fixes → {imp_acc}/{len(labels)}")
        elif n_fix == 0:
            break

    return best_model


print("Self-training refinement functions defined.")

In [ ]:
# ── Full Pipeline Evaluator ────────────────────────────
def evaluate(model, images, queries, labels=None, desc='Evaluating'):
    """
    Full pipeline: perceive → parse → execute → compare.
    Returns (predictions, accuracy).
    """
    predictions, correct = [], 0
    for i in tqdm(range(len(images)), desc=desc, leave=False):
        objs = perceive_image(model, images[i])
        pred = bool(execute(parse_query(queries[i]), objs))
        predictions.append(pred)
        if labels is not None and pred == labels[i]:
            correct += 1
    acc = correct / len(images) if labels is not None else None
    return predictions, acc


print("evaluate() defined.")

---
## Section 4 — Results Across All 4 Dataset Sizes

Now we run the full hybrid pipeline on every training split. For each size
we:
1. Generate heuristic cell labels
2. Refine them via self-training with query feedback
3. Train a final CNN on the refined labels
4. Evaluate on the shared 1024-image test set

The key question: **how does test accuracy scale with training set size?**
More training data means more cells to learn from *and* more query–answer
pairs to correct mislabelled cells during self-training.

In [ ]:
# ── Main Training Loop Over All Sizes ──────────────────
all_results = {}

for sz in SIZES:
    print(f"\n{'='*60}")
    print(f"  Size: {sz.upper()}")
    print(f"{'='*60}")

    # 1. Load data
    imgs, queries, labels = load_split(sz)
    print(f"  Images: {len(imgs)}")

    # 2. Heuristic label cells
    cells, shapes, colors, ici = _build_cell_data(imgs)
    print(f"  Non-empty cells: {len(cells)}")
    print(f"  Shapes: { {s: int((shapes==i).sum()) for i,s in enumerate(SHAPES)} }")
    print(f"  Colors: { {c: int((colors==i).sum()) for i,c in enumerate(COLORS)} }")

    # 3. Self-training refinement
    print(f"\n  Self-training refinement:")
    parsed = [parse_query(q) for q in queries]
    _ = refine_labels(cells, shapes, colors, ici, parsed, labels,
                      rounds=3, cnn_epochs=25)

    # 4. Final CNN training on refined labels
    print(f"\n  Final CNN training (30 epochs):")
    model, history = train_cnn(cells, shapes, colors, epochs=30, batch_size=256)

    # 5. Plot training history
    plot_training_history(history, title=f'{sz.upper()} — CNN Training')

    # 6. Evaluate on test set
    test_preds, test_acc = evaluate(model, test_images, test_queries,
                                   test_labels, desc=f'{sz} test')
    print(f"\n  Test accuracy: {test_acc:.4f} ({int(test_acc*len(test_images))}/{len(test_images)})")

    # Store results
    all_results[sz] = {
        'model': model,
        'history': history,
        'test_preds': test_preds,
        'test_acc': test_acc,
        'n_images': len(imgs),
        'n_cells': len(cells),
        'final_shape_acc': history['shape_acc'][-1],
        'final_color_acc': history['color_acc'][-1],
    }

print(f"\n{'='*60}")
print("  All sizes complete!")
print(f"{'='*60}")

In [ ]:
# ── Comparison Charts ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart: test accuracy per size
accs = [all_results[sz]['test_acc'] for sz in SIZES]
bars = axes[0].bar(SIZES, accs, color=['#3498db', '#2ecc71', '#e67e22', '#e74c3c'])
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Test Accuracy by Training Size')
axes[0].set_ylim(0, 1.05)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{acc:.3f}', ha='center', fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Grouped bar chart: shape acc + color acc per size
x = np.arange(len(SIZES))
w = 0.35
s_accs = [all_results[sz]['final_shape_acc'] for sz in SIZES]
c_accs = [all_results[sz]['final_color_acc'] for sz in SIZES]
axes[1].bar(x - w/2, s_accs, w, label='Shape Acc', color='#9b59b6')
axes[1].bar(x + w/2, c_accs, w, label='Color Acc', color='#1abc9c')
axes[1].set_xticks(x); axes[1].set_xticklabels(SIZES)
axes[1].set_ylabel('Accuracy'); axes[1].set_title('CNN Cell Accuracy (Shape vs Color)')
axes[1].legend(); axes[1].set_ylim(0, 1.05); axes[1].grid(True, alpha=0.3, axis='y')

# Text summary table
axes[2].axis('off')
table_data = [['Size', 'Images', 'Cells', 'Test Acc', 'Correct']]
for sz in SIZES:
    r = all_results[sz]
    table_data.append([sz, str(r['n_images']), str(r['n_cells']),
                       f"{r['test_acc']:.4f}",
                       f"{int(r['test_acc']*len(test_images))}/{len(test_images)}"])
table = axes[2].table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1.2, 1.8)
# Header styling
for j in range(5):
    table[0, j].set_facecolor('#34495e')
    table[0, j].set_text_props(color='white', fontweight='bold')
axes[2].set_title('Summary Table', fontsize=13, pad=20)

plt.tight_layout()
plt.show()

---
## Section 5 — Detailed Metrics & Analysis

Accuracy alone can be misleading — especially with roughly balanced classes.
Here we compute **precision, recall, F1, and confusion matrices** for each
training size, then compare them side-by-side to understand where the pipeline
succeeds and where it still makes mistakes.

All metrics are computed from scratch using only numpy — no sklearn.

In [ ]:
# ── Metrics Computation ────────────────────────────────
def compute_metrics(predictions, labels):
    """Compute TP, FP, FN, TN, Precision, Recall, F1, Accuracy from scratch."""
    preds = np.array(predictions, dtype=bool)
    labs  = np.array(labels, dtype=bool)
    TP = int(np.sum(preds & labs))
    FP = int(np.sum(preds & ~labs))
    FN = int(np.sum(~preds & labs))
    TN = int(np.sum(~preds & ~labs))
    precision = TP / (TP + FP + 1e-8)
    recall    = TP / (TP + FN + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy  = (TP + TN) / len(labs)
    return {
        'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
        'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy,
    }


# Compute metrics for every size
all_metrics = {}
for sz in SIZES:
    m = compute_metrics(all_results[sz]['test_preds'], test_labels)
    all_metrics[sz] = m
    print(f"\n  {sz.upper()}:")
    print(f"    TP={m['TP']:>4}  FP={m['FP']:>4}  FN={m['FN']:>4}  TN={m['TN']:>4}")
    print(f"    Precision={m['precision']:.4f}  Recall={m['recall']:.4f}  "
          f"F1={m['f1']:.4f}  Accuracy={m['accuracy']:.4f}")

In [ ]:
# ── Per-Size Detailed Figures ──────────────────────────
for sz in SIZES:
    m = all_metrics[sz]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(f'{sz.upper()} — Detailed Test Metrics', fontsize=14, y=1.02)

    # 1. Confusion matrix heatmap
    cm = np.array([[m['TP'], m['FP']], [m['FN'], m['TN']]])
    im = axes[0].imshow(cm, cmap='Blues', aspect='auto')
    axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Pred True', 'Pred False'])
    axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['Actual True', 'Actual False'])
    axes[0].set_title('Confusion Matrix')
    for (r, c), val in np.ndenumerate(cm):
        axes[0].text(c, r, str(val), ha='center', va='center',
                     fontsize=16, fontweight='bold',
                     color='white' if val > cm.max()/2 else 'black')

    # 2. Bar chart of Precision / Recall / F1 / Accuracy
    metric_names = ['Precision', 'Recall', 'F1', 'Accuracy']
    metric_vals  = [m['precision'], m['recall'], m['f1'], m['accuracy']]
    bars = axes[1].bar(metric_names, metric_vals,
                       color=['#e74c3c', '#2ecc71', '#3498db', '#9b59b6'])
    axes[1].set_ylim(0, 1.05); axes[1].set_title('Metric Summary')
    axes[1].grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, metric_vals):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.3f}', ha='center', fontsize=10)

    # 3. Operating point on ROC unit square
    fpr = m['FP'] / (m['FP'] + m['TN'] + 1e-8)
    tpr = m['TP'] / (m['TP'] + m['FN'] + 1e-8)
    axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
    axes[2].scatter([fpr], [tpr], s=150, c='red', zorder=5, edgecolors='black')
    axes[2].annotate(f'({fpr:.3f}, {tpr:.3f})', (fpr, tpr),
                     textcoords='offset points', xytext=(10, -15), fontsize=10)
    axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
    axes[2].set_title('Operating Point'); axes[2].set_xlim(-0.05, 1.05)
    axes[2].set_ylim(-0.05, 1.05); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    axes[2].set_aspect('equal')

    # 4. Text summary box
    axes[3].axis('off')
    summary_text = (
        f"Accuracy:  {m['accuracy']:.4f}\n"
        f"Precision: {m['precision']:.4f}\n"
        f"Recall:    {m['recall']:.4f}\n"
        f"F1 Score:  {m['f1']:.4f}\n"
        f"\nConfusion:\n"
        f"  TP={m['TP']}  FP={m['FP']}\n"
        f"  FN={m['FN']}  TN={m['TN']}"
    )
    axes[3].text(0.1, 0.5, summary_text, transform=axes[3].transAxes,
                fontsize=13, verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))
    axes[3].set_title('Numeric Summary')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Combined Comparison Across All Sizes ───────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# 1. Grouped bar chart: P, R, F1, Acc — one group per metric, one bar per size
metric_keys = ['precision', 'recall', 'f1', 'accuracy']
metric_labels = ['Precision', 'Recall', 'F1', 'Accuracy']
x = np.arange(len(metric_keys))
w = 0.18
size_colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

for j, sz in enumerate(SIZES):
    vals = [all_metrics[sz][k] for k in metric_keys]
    ax1.bar(x + j * w, vals, w, label=sz, color=size_colors[j])

ax1.set_xticks(x + 1.5 * w)
ax1.set_xticklabels(metric_labels)
ax1.set_ylabel('Score'); ax1.set_title('Metrics Comparison Across Sizes')
ax1.legend(); ax1.set_ylim(0, 1.1); ax1.grid(True, alpha=0.3, axis='y')

# 2. Line plot: F1 vs dataset size
f1_scores = [all_metrics[sz]['f1'] for sz in SIZES]
n_images  = [all_results[sz]['n_images'] for sz in SIZES]
ax2.plot(range(len(SIZES)), f1_scores, 'bo-', markersize=10, linewidth=2)
ax2.set_xticks(range(len(SIZES)))
ax2.set_xticklabels([f'{sz}\n({n})' for sz, n in zip(SIZES, n_images)])
ax2.set_ylabel('F1 Score'); ax2.set_xlabel('Training Size')
ax2.set_title('F1 Score vs Training Data Size')
ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 1.05)
for i, (sz, f1) in enumerate(zip(SIZES, f1_scores)):
    ax2.annotate(f'{f1:.3f}', (i, f1), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=11)

plt.tight_layout()
plt.show()

---
## Section 6 — Save Outputs

We save the test predictions for every training size as well as the best
model's weights, so the results are fully reproducible and can be submitted
or evaluated offline.

In [ ]:
# ── Save Predictions & Best Model ──────────────────────
# Find the best-performing size
best_size = max(SIZES, key=lambda sz: all_results[sz]['test_acc'])
print(f"Best performing size: {best_size} (test acc = {all_results[best_size]['test_acc']:.4f})\n")

# Save predictions for every size
for sz in SIZES:
    fname = f'predictions_{sz}.txt'
    with open(fname, 'w') as f:
        for p in all_results[sz]['test_preds']:
            f.write('true\n' if p else 'false\n')
    print(f"  Saved {fname}")

# Save best model
model_fname = f'cell_classifier_{best_size}.pth'
torch.save(all_results[best_size]['model'].state_dict(), model_fname)
print(f"  Saved {model_fname}")

print(f"\nDone! All outputs saved.")